In [26]:
import torch
import torch.nn as nn

In [27]:
x = torch.tensor([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [1, 3, 5, 7, 9],
    [0, 2, 4, 6, 8],
    [4, 5, 6, 7, 8],
    [5, 4, 3, 2, 1],
    [6, 5, 4, 3, 2],
    [9, 7, 5, 3, 1],
    [3, 1, 4, 1, 5],
    [2, 7, 1, 8, 2],
    [4, 4, 4, 4, 4],
], dtype=torch.float32)
print(f"x: {x}")
# 1 if increasing, 0 if decreasing (mixed or flat also count as 0)
y = torch.tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=torch.float32).view(-1, 1)
print(f"y: {y}")
neurons = 4

x: tensor([[1., 2., 3., 4., 5.],
        [2., 3., 4., 5., 6.],
        [3., 4., 5., 6., 7.],
        [1., 3., 5., 7., 9.],
        [0., 2., 4., 6., 8.],
        [4., 5., 6., 7., 8.],
        [5., 4., 3., 2., 1.],
        [6., 5., 4., 3., 2.],
        [9., 7., 5., 3., 1.],
        [3., 1., 4., 1., 5.],
        [2., 7., 1., 8., 2.],
        [4., 4., 4., 4., 4.]])
y: tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [28]:
class MultiLayerPerceptron(nn.Module):
    def __init__(self, input_size, output_size, hidden_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=input_size, out_features=hidden_size),
            nn.ReLU(),
            nn.Linear(in_features=hidden_size, out_features=output_size),
        )

    def forward(self, x):
        return self.layers(x)

In [29]:
# model initialization
model = MultiLayerPerceptron(input_size=x.shape[-1], output_size=y.shape[-1], hidden_size=neurons)
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

layers.0.weight: tensor([[-0.0039,  0.1280, -0.2362,  0.1147,  0.0479],
        [ 0.1292,  0.2493, -0.3144,  0.0628,  0.0056],
        [ 0.1438, -0.2788, -0.2990, -0.1130, -0.1319],
        [ 0.4293, -0.3547,  0.2217, -0.0449, -0.2802]])
layers.0.bias: tensor([ 0.4205,  0.2643, -0.2390, -0.0850])
layers.2.weight: tensor([[ 0.1074, -0.0723,  0.0077,  0.0778]])
layers.2.bias: tensor([0.0064])


In [30]:
epochs = 500
for epoch in range(epochs):
    # forward pass
    z = model(x)
    # calculate loss
    loss = nn.BCEWithLogitsLoss()(input=z, target=y)
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss: {loss.item()}")
        for name, param in model.named_parameters():
            print(f"{name}.grad: {param.grad}")
    # update parameters
    optimizer.step()

epoch 0 loss: 0.6883365511894226
layers.0.weight.grad: tensor([[ 0.0718,  0.0418, -0.0387, -0.0596, -0.1401],
        [-0.0483, -0.0338,  0.0146,  0.0229,  0.0713],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0768,  0.0568,  0.0533,  0.0300,  0.0298]])
layers.0.bias.grad: tensor([-0.0030, -0.0008,  0.0000,  0.0133])
layers.2.weight.grad: tensor([[-0.0054,  0.2442,  0.0000,  0.1774]])
layers.2.bias.grad: tensor([0.0143])
epoch 50 loss: 0.16442202031612396
layers.0.weight.grad: tensor([[ 0.0728, -0.0006,  0.0342, -0.0614,  0.0015],
        [-0.0294, -0.0275,  0.0051,  0.0015,  0.0340],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000]])
layers.0.bias.grad: tensor([ 0.0141, -0.0058,  0.0000,  0.0000])
layers.2.weight.grad: tensor([[-0.1124,  0.1127,  0.0000,  0.0000]])
layers.2.bias.grad: tensor([0.0310])
epoch 100 loss: 0.05292654410004616
layers.0.weight.grad: tensor([[-0.0036, -0.0028, -0.0324, -0.0261, -

In [31]:
# prediction
z = model(x)
# activation function (value > 0 is 1, else 0)
y_pred = (z > 0).float()
print(f"y_pred: {y_pred.view(-1)}")
print(f"y:      {y.view(-1)}")

y_pred: tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
y:      tensor([1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.])
